# analysis_manifest_public_surface_v1

目标：验证旧 analysis sidecar 通过 `src.analysis.manifest_backfill` 升级为 manifest 后，dashboard 只能读取 manifest 声明的 public artifact，并且新的模型差异解释、segment matrix、review briefing 不会在 backfill 中丢失。

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd()
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / 'src').exists():
        REPO_ROOT = candidate
        sys.path.insert(0, str(candidate))
        break

from src.analysis import backfill_analysis_manifest
from src.dashboard import load_model_matrix_dashboard

NOTEBOOK_DIR = REPO_ROOT / 'research/analysis_manifest_public_surface_v1'
NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)
legacy_card = NOTEBOOK_DIR / 'legacy_card'
manifest_only_card = NOTEBOOK_DIR / 'manifest_only_card'
for card in (legacy_card, manifest_only_card):
    card.mkdir(parents=True, exist_ok=True)


In [2]:
def write_base_card(card: Path, *, status: str) -> None:
    (card / 'analysis_summary.json').write_text(
        json.dumps(
            {
                'schema_version': 'analysis.report.v1',
                'status': status,
                'start': '2026-01-01',
                'end': '2026-06-09',
                'model_count': 2,
                'note': '中文 review：manifest public surface 验证卡片。',
            },
            ensure_ascii=False,
            indent=2,
        ),
        encoding='utf-8',
    )
    pd.DataFrame(
        [
            {'rank': 1, 'model': 'ridge_mse', 'return_pct': 3.0, 'relative_return_pct': 2.0, 'stability_score': 4.0},
            {'rank': 2, 'model': 'gcn_mse', 'return_pct': 1.0, 'relative_return_pct': 0.5, 'stability_score': 1.5},
        ]
    ).to_csv(card / 'model_comparison.csv', index=False)
    pd.DataFrame([{'model': 'ridge_mse', 'regime': 'bull', 'return_pct': 3.0}]).to_csv(
        card / 'regime_metrics.csv', index=False
    )
    pd.DataFrame([{'model': 'ridge_mse', 'window': 'test', 'return_pct': 3.0}]).to_csv(
        card / 'window_metrics.csv', index=False
    )
    pd.DataFrame([{'model': 'ridge_mse', 'segment_type': 'all', 'segment': 'all', 'diagnosis': '基准模型。'}]).to_csv(
        card / 'model_effect_deltas.csv', index=False
    )
    pd.DataFrame([{'model': 'ridge_mse', 'paper_fidelity': 'native_library'}]).to_csv(
        card / 'model_capabilities.csv', index=False
    )
    pd.DataFrame([{'model': 'ridge_mse', 'prediction': 0.1}]).to_csv(card / 'predictions.csv', index=False)
    pd.DataFrame([{'model': 'ridge_mse', 'nav': 1.0}]).to_csv(card / 'daily_backtest.csv', index=False)


write_base_card(legacy_card, status='legacy_public_surface_completed')
write_base_card(manifest_only_card, status='manifest_only_completed')


In [3]:
pd.DataFrame(
    [
        {
            'schema_version': 'analysis.segment_effect_matrix.v1',
            'segment_type': 'regime',
            'segment': 'bull',
            'baseline_model': 'ridge_mse',
            'best_relative_return_model': 'ridge_mse',
            'worst_relative_return_model': 'gcn_mse',
        }
    ]
).to_csv(legacy_card / 'segment_effect_matrix.csv', index=False)
pd.DataFrame(
    [
        {
            'schema_version': 'analysis.model_difference_explanation.v1',
            'priority_rank': 1,
            'segment_type': 'regime',
            'segment': 'bull',
            'baseline_model': 'ridge_mse',
            'winner_model': 'ridge_mse',
            'loser_model': 'gcn_mse',
            'explanation_signal': 'baseline_defends',
            'decision_hint': 'block_or_rework_loser',
            'review_note': '中文 review：legacy backfill 后仍能暴露模型差异解释。',
        }
    ]
).to_csv(legacy_card / 'model_difference_explanations.csv', index=False)
pd.DataFrame(
    [
        {
            'schema_version': 'analysis.model_review_briefing.v1',
            'priority_rank': 1,
            'model': 'gcn_mse',
            'baseline_model': 'ridge_mse',
            'segment_type': 'regime',
            'segment': 'bull',
            'review_signal': 'weakness',
            'decision_hint': 'block_or_rework',
            'review_note': '中文 review：legacy backfill 后仍能暴露 review briefing。',
        }
    ]
).to_csv(legacy_card / 'model_review_briefing.csv', index=False)

legacy_manifest = backfill_analysis_manifest(
    legacy_card,
    dataset_name='analysis_manifest_public_surface_v1',
    universe=('SOXL', 'SOXX'),
    calendar_path='data/qlib/soxl/calendars/day.txt',
    data_paths=(str(legacy_card),),
    overwrite=True,
)
legacy_dashboard = load_model_matrix_dashboard(legacy_card)

assert legacy_manifest['paths']['segment_effect_matrix'] == 'segment_effect_matrix.csv'
assert legacy_manifest['paths']['model_difference_explanations'] == 'model_difference_explanations.csv'
assert legacy_manifest['paths']['model_review_briefing'] == 'model_review_briefing.csv'
assert 'predictions' not in legacy_manifest['paths']
assert 'daily_backtest' not in legacy_manifest['paths']
assert legacy_dashboard['model_difference_explanations'][0]['explanation_signal'] == 'baseline_defends'
assert legacy_dashboard['model_review_briefing'][0]['review_signal'] == 'weakness'
legacy_manifest['paths']


{'model_capabilities': 'model_capabilities.csv',
 'model_comparison': 'model_comparison.csv',
 'model_difference_explanations': 'model_difference_explanations.csv',
 'model_effect_deltas': 'model_effect_deltas.csv',
 'model_review_briefing': 'model_review_briefing.csv',
 'regime_metrics': 'regime_metrics.csv',
 'segment_effect_matrix': 'segment_effect_matrix.csv',
 'summary': 'analysis_summary.json',
 'window_metrics': 'window_metrics.csv'}

In [4]:
pd.DataFrame(
    [
        {
            'schema_version': 'analysis.model_difference_explanation.v1',
            'priority_rank': 1,
            'segment_type': 'regime',
            'segment': 'bull',
            'baseline_model': 'ridge_mse',
            'winner_model': 'ridge_mse',
            'loser_model': 'gcn_mse',
            'explanation_signal': 'baseline_defends',
            'decision_hint': 'block_or_rework_loser',
        }
    ]
).to_csv(manifest_only_card / 'model_difference_explanations.csv', index=False)
(manifest_only_card / 'analysis_manifest.json').write_text(
    json.dumps(
        {
            'artifact_id': 'manifest_only_card',
            'artifact_type': 'analysis_report',
            'producer': 'src.analysis.exports',
            'schema_version': 'artifact.v1',
            'paths': {
                'summary': 'analysis_summary.json',
                'model_comparison': 'model_comparison.csv',
                'regime_metrics': 'regime_metrics.csv',
                'window_metrics': 'window_metrics.csv',
            },
            'metadata': {'analysis_report_schema_version': 'analysis.report.v1'},
            'manifest_hash': 'm' * 64,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding='utf-8',
)
manifest_only_dashboard = load_model_matrix_dashboard(manifest_only_card)
assert manifest_only_dashboard['manifest_source']['kind'] == 'analysis_manifest'
assert manifest_only_dashboard['model_difference_explanations'] == []
assert 'model_difference_explanations' not in manifest_only_dashboard['artifacts']
manifest_only_dashboard['artifacts']


{'summary': '/Users/rziiiii/dev/workspace/research/analysis_manifest_public_surface_v1/manifest_only_card/analysis_summary.json',
 'model_comparison': '/Users/rziiiii/dev/workspace/research/analysis_manifest_public_surface_v1/manifest_only_card/model_comparison.csv',
 'regime_metrics': '/Users/rziiiii/dev/workspace/research/analysis_manifest_public_surface_v1/manifest_only_card/regime_metrics.csv',
 'window_metrics': '/Users/rziiiii/dev/workspace/research/analysis_manifest_public_surface_v1/manifest_only_card/window_metrics.csv',
 'analysis_manifest': '/Users/rziiiii/dev/workspace/research/analysis_manifest_public_surface_v1/manifest_only_card/analysis_manifest.json'}

In [5]:
summary = {
    'schema_version': 'research.analysis_manifest_public_surface_v1.summary.v1',
    'status': 'completed',
    'artifact_id': 'analysis_manifest_public_surface_v1',
    'legacy_manifest_hash': legacy_manifest['manifest_hash'],
    'legacy_public_paths': sorted(legacy_manifest['paths']),
    'dashboard_manifest_source': legacy_dashboard['manifest_source']['kind'],
    'model_difference_rows': len(legacy_dashboard['model_difference_explanations']),
    'model_review_rows': len(legacy_dashboard['model_review_briefing']),
    'manifest_only_unmanifested_difference_rows': len(manifest_only_dashboard['model_difference_explanations']),
    'private_paths_exposed': sorted(
        key for key in legacy_manifest['paths'] if key in {'predictions', 'daily_backtest', 'adapter_predictions'}
    ),
    'conclusion': '中文结论：legacy backfill 已暴露当前 public analysis 表；dashboard 在有 manifest 时不再读取未声明固定 sidecar。',
}
(NOTEBOOK_DIR / 'analysis_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
(NOTEBOOK_DIR / 'dashboard_export.json').write_text(json.dumps(legacy_dashboard, ensure_ascii=False, indent=2), encoding='utf-8')
summary


{'schema_version': 'research.analysis_manifest_public_surface_v1.summary.v1',
 'status': 'completed',
 'artifact_id': 'analysis_manifest_public_surface_v1',
 'legacy_manifest_hash': '87cd01593e8f23f147eb762d69248da8f62b036098b76e8ce1aa3d19d97b49f5',
 'legacy_public_paths': ['model_capabilities',
  'model_comparison',
  'model_difference_explanations',
  'model_effect_deltas',
  'model_review_briefing',
  'regime_metrics',
  'segment_effect_matrix',
  'summary',
  'window_metrics'],
 'dashboard_manifest_source': 'analysis_manifest',
 'model_difference_rows': 1,
 'model_review_rows': 1,
 'manifest_only_unmanifested_difference_rows': 0,
 'private_paths_exposed': [],
 'conclusion': '中文结论：legacy backfill 已暴露当前 public analysis 表；dashboard 在有 manifest 时不再读取未声明固定 sidecar。'}

## 中文结论

- 做了什么：验证 `src.analysis.manifest_backfill` 会把当前 analysis public sidecars 写入 `analysis_manifest.json`，并验证 dashboard 在存在 manifest 时只读取 manifest 声明路径。
- 核心指标：legacy manifest 暴露 `segment_effect_matrix`、`model_difference_explanations`、`model_review_briefing`；dashboard 读取到 1 条模型差异解释和 1 条 review briefing；未声明的固定 `model_difference_explanations.csv` 在 manifest-only 卡片中读取行数为 0。
- regime 差异：示例使用 `bull` regime 验证模型差异表和 review briefing 能随 manifest 进入 dashboard；本轮不重新评估真实市场 regime。
- 是否晋升：不晋升策略，仅晋升框架边界；该能力服务旧 analysis card 迁移和 schedule auto-backfill。
- 风险：已有未带 manifest 的 legacy card 仍允许固定文件 fallback；一旦生成 manifest，缺少 public key 会导致 dashboard 看不到对应表，需要通过 backfill 或重新导出修复。
- 下一步：继续把更多 production/research 卡片统一到 manifest-first，并减少 dashboard 对无 manifest fixed sidecar 的兼容路径依赖。